# 02 — Build Silver | Olist (versión consolidada)

Única implementación activa: nueve entidades, reglas de calidad, `MERGE` incremental, rechazos y auditoría de duplicados **por lote**. Las antiguas celdas de escritura duplicadas fueron retiradas.

Las columnas técnicas `__ingest_*` se utilizan solamente para auditoría; no modifican el esquema analítico de Silver.


In [0]:
# CELDA 1 — Configuración y controles de existencia
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType
from delta.tables import DeltaTable
from datetime import datetime, timezone
import uuid

CATALOG = "workspace"
SCHEMA = "olist_lakehouse"
# Olist publica timestamps sin zona explícita: fijamos la zona de sesión
# para no obtener interpretaciones diferentes entre ejecuciones.
# UTC es una convención técnica, NO una afirmación sobre la zona real de origen.
spark.conf.set("spark.sql.session.timeZone", "UTC")
RUN_ID = str(uuid.uuid4())

TABLES = [
    "customers", "sellers", "category_translation", "orders",
    "order_items", "order_payments", "order_reviews", "products", "geolocation"
]
KEYS = {
    "customers": ["customer_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name"],
    "orders": ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id", "order_id"],
    "products": ["product_id"],
    "geolocation": ["geolocation_zip_code_prefix"]
}

BRONZE_ALIASES = {"order_reviews": "order_reviews", "category_translation": "category_translation"}

def bronze_name(table):
    return f"{CATALOG}.{SCHEMA}.bronze_{BRONZE_ALIASES.get(table, table)}"

def silver_name(table):
    return f"{CATALOG}.{SCHEMA}.silver_{table}"

for table in TABLES:
    assert spark.catalog.tableExists(bronze_name(table)), f"No existe {bronze_name(table)}"

print("Bronze disponible. RUN_ID:", RUN_ID)

# Si se invoca desde Jobs, puede recibirse el lote de Bronze como parámetro.
# En ejecución interactiva se toma el último lote registrado en auditoría.
dbutils.widgets.text("bronze_batch_id", "")
BRONZE_BATCH_ID = dbutils.widgets.get("bronze_batch_id").strip()
INGEST_META = {"__ingest_batch_id", "__ingest_file", "__ingest_file_sha256", "__ingest_at"}
bronze_audit_table = f"{CATALOG}.{SCHEMA}.bronze_ingestion_audit"
if not BRONZE_BATCH_ID and spark.catalog.tableExists(bronze_audit_table):
    latest = (spark.table(bronze_audit_table)
        .groupBy("batch_id").agg(F.max("ingestion_timestamp").alias("last_time"))
        .orderBy(F.desc("last_time")).limit(1).collect())
    BRONZE_BATCH_ID = latest[0]["batch_id"] if latest else ""
print("Lote Bronze observado por Silver:", BRONZE_BATCH_ID or "SIN LOTE")


In [0]:
# CELDA 2 — Funciones compartidas: tipificación segura, calidad, upsert y auditoría

AUDIT = f"{CATALOG}.{SCHEMA}.silver_pipeline_audit"
REJECTS = f"{CATALOG}.{SCHEMA}.silver_quality_rejects"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {REJECTS} (
  source_table STRING,
  source_record_hash STRING,
  reject_reason STRING,
  raw_payload STRING,
  first_detected_at TIMESTAMP
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {AUDIT} (
  run_id STRING,
  source_table STRING,
  source_rows BIGINT,
  rejected_rows BIGINT,
  valid_key_rows BIGINT,
  duplicates_resolved BIGINT,
  silver_rows BIGINT,
  warning_rows BIGINT,
  run_at TIMESTAMP
) USING DELTA
""")


def q(name):
    return "`" + name.replace("`", "``") + "`"


def normalize_text(df, columns, mode="trim"):
    for c in columns:
        if mode == "upper":
            value = F.upper(F.trim(F.col(c)))
        elif mode == "lower":
            value = F.lower(F.trim(F.col(c)))
        elif mode == "city":
            value = F.initcap(F.trim(F.col(c)))
        else:
            value = F.trim(F.col(c))
        df = df.withColumn(c, F.when(F.length(value) == 0, F.lit(None)).otherwise(value))
    return df


def safe_cast(df, types):
    for c, sql_type in types.items():
        df = df.withColumn(c, F.expr(f"try_cast({q(c)} AS {sql_type})"))
    return df


def invalid_conversion(raw, parsed):
    # raw must be supplied before replacement if calling this function directly;
    # the main transformations use the original raw fields as __raw_<column>.
    return raw.isNotNull() & (F.trim(raw.cast("string")) != "") & parsed.isNull()


def read_bronze(table):
    df = spark.table(bronze_name(table))
    raw_columns = sorted(c for c in df.columns if c not in INGEST_META)
    raw_payload = F.to_json(
        F.struct(*[F.col(q(c)).alias(c) for c in raw_columns]),
        options={"ignoreNullFields": "false"}
    )
    return (
        df.withColumn("__ingest_batch_id", F.col("__ingest_batch_id") if "__ingest_batch_id" in df.columns else F.lit(None).cast("string"))
          .withColumn("__raw_payload", raw_payload)
          .withColumn("source_record_hash", F.sha2(F.col("__raw_payload"), 256))
          .drop(*[c for c in INGEST_META if c != "__ingest_batch_id" and c in df.columns])
    )


def add_quality(df, rules):
    # rules: list of (issue_code, boolean Spark Column); False/NULL = no issue.
    flags = [F.when(condition, F.lit(name)) for name, condition in rules]
    df = df.withColumn("dq_issues", F.concat_ws("|", *flags))
    df = df.withColumn(
        "dq_issue_count",
        F.when(F.col("dq_issues") == "", F.lit(0))
         .otherwise(F.size(F.split(F.col("dq_issues"), r"\|")))
    )
    df = df.withColumn(
        "dq_status", F.when(F.col("dq_issue_count") == 0, F.lit("OK"))
                       .otherwise(F.lit("WARN"))
    )
    return df


def key_invalid(df, keys):
    invalid_parts = []
    for key in keys:
        dtype = df.schema[key].dataType
        if isinstance(dtype, StringType):
            invalid_parts.append(F.col(key).isNull() | (F.trim(F.col(key)) == ""))
        else:
            invalid_parts.append(F.col(key).isNull())
    # IDs secuenciales Olist deben ser > 0
    for key in ("order_item_id", "payment_sequential"):
        if key in keys:
            invalid_parts.append(F.col(key) <= F.lit(0))
    result = invalid_parts[0]
    for condition in invalid_parts[1:]:
        result = result | condition
    return result


def save_rejects(invalid_df, table):
    rejected = invalid_df.select(
        F.lit(table).alias("source_table"),
        F.col("source_record_hash"),
        F.lit("NULL_OR_INVALID_BUSINESS_KEY").alias("reject_reason"),
        F.col("__raw_payload").alias("raw_payload"),
        F.current_timestamp().alias("first_detected_at")
    ).dropDuplicates(["source_table", "source_record_hash"])

    if rejected.limit(1).count() > 0:
        (
            DeltaTable.forName(spark, REJECTS).alias("t")
            .merge(
                rejected.alias("s"),
                "t.source_table = s.source_table AND "
                "t.source_record_hash = s.source_record_hash"
            )
            .whenNotMatchedInsertAll()
            .execute()
        )


def choose_one_per_key(df, keys):
    # Prefer fewer quality issues, then more populated values, then stable hash.
    payload_cols = [c for c in df.columns
                    if c not in {"__raw_payload", "source_record_hash",
                                 "dq_status", "dq_issues", "dq_issue_count"}
                    and not c.startswith("__raw_")]
    completeness = F.lit(0)
    for c in payload_cols:
        completeness = completeness + F.when(F.col(q(c)).isNotNull(), 1).otherwise(0)

    window = Window.partitionBy(*[F.col(q(c)) for c in keys]).orderBy(
        F.col("dq_issue_count").asc(),
        F.col("__completeness").desc(),
        F.col("source_record_hash").asc()
    )
    return (
        df.withColumn("__completeness", completeness)
          .withColumn("__rank", F.row_number().over(window))
          .filter(F.col("__rank") == 1)
          .drop("__rank", "__completeness")
    )


def add_silver_hash(df):
    fields = sorted(c for c in df.columns if not c.startswith("__"))
    payload = F.to_json(
        F.struct(*[F.col(q(c)).alias(c) for c in fields]),
        options={"ignoreNullFields": "false"}
    )
    return df.withColumn("silver_row_hash", F.sha2(payload, 256))


def ensure_target_schema(df, table):
    dest = silver_name(table)
    if not spark.catalog.tableExists(dest):
        schema_sql = ",\n".join(
            f"{q(f.name)} {f.dataType.simpleString()}" for f in df.schema.fields
        )
        spark.sql(f"CREATE TABLE IF NOT EXISTS {dest} ({schema_sql}) USING DELTA")
    else:
        existing = {f.name: f.dataType for f in spark.table(dest).schema.fields}
        new_fields = []
        for field in df.schema.fields:
            if field.name in existing:
                if existing[field.name] != field.dataType:
                    raise TypeError(
                        f"{dest}.{field.name}: tipo existente {existing[field.name]} "
                        f"difiere del nuevo {field.dataType}. Revisar migración; no sobrescribir."
                    )
            else:
                new_fields.append(f"{q(field.name)} {field.dataType.simpleString()}")
        if new_fields:
            spark.sql(f"ALTER TABLE {dest} ADD COLUMNS ({', '.join(new_fields)})")


def upsert_silver(df, table, keys):
    dest = silver_name(table)
    ensure_target_schema(df, table)
    cond = " AND ".join(f"t.{q(k)} = s.{q(k)}" for k in keys)
    updates = {c: f"s.{q(c)}" for c in df.columns}

    (
        DeltaTable.forName(spark, dest).alias("t")
        .merge(df.alias("s"), cond)
        .whenMatchedUpdate(
            condition="t.silver_row_hash IS NULL OR "
                      "t.silver_row_hash <> s.silver_row_hash",
            set=updates
        )
        .whenNotMatchedInsert(values=updates)
        .execute()
    )


def audit_table(table, source_rows, rejected, valid, resolved, silver_rows, warnings):
    from pyspark.sql.types import (
        StructType, StructField, StringType, LongType, TimestampType
    )
    schema = StructType([
        StructField("run_id", StringType()),
        StructField("source_table", StringType()),
        StructField("source_rows", LongType()),
        StructField("rejected_rows", LongType()),
        StructField("valid_key_rows", LongType()),
        StructField("duplicates_resolved", LongType()),
        StructField("silver_rows", LongType()),
        StructField("warning_rows", LongType()),
        StructField("run_at", TimestampType())
    ])
    audit_df = spark.createDataFrame([(
        RUN_ID, table, int(source_rows), int(rejected), int(valid),
        int(resolved), int(silver_rows), int(warnings), datetime.now(timezone.utc)
    )], schema=schema)
    (
        DeltaTable.forName(spark, AUDIT).alias("t")
        .merge(audit_df.alias("s"),
               "t.run_id = s.run_id AND t.source_table = s.source_table")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )



# Incidencias del lote NUEVO: tabla adicional, sin modificar la auditoría Silver histórica.
DUPLICATE_AUDIT = f"{CATALOG}.{SCHEMA}.silver_duplicate_alerts"
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DUPLICATE_AUDIT} (
    silver_run_id STRING, bronze_batch_id STRING, source_table STRING,
    rows_received BIGINT, repeated_key_rows BIGINT,
    exact_duplicate_rows BIGINT, conflicting_version_rows BIGINT,
    new_key_rows BIGINT, alert_level STRING, recorded_at TIMESTAMP
) USING DELTA
""")

def audit_duplicate_batch(valid, table, keys, by_payload_only=False):
    """Cuenta incidencias de la ingesta actual sin volver a reportar históricas.
    El campo __ingest_batch_id es una columna interna de Bronze (no se publica en Silver).
    Geolocation compara payloads (no ZIP): ZIP repetido es una observación legítima.
    """
    from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType
    received = valid.filter(F.col("__ingest_batch_id") == F.lit(BRONZE_BATCH_ID)) if BRONZE_BATCH_ID else valid.limit(0)
    n = received.count()
    if not n:
        metrics = (0, 0, 0, 0, 0)
    else:
        previous = valid.filter(F.col("__ingest_batch_id").isNull() |
                                (F.col("__ingest_batch_id") != F.lit(BRONZE_BATCH_ID)))
        key_cols = ["source_record_hash"] if by_payload_only else keys
        # Evitar columnas duplicadas en el DataFrame de entrada.
        incoming = received.select(*list(dict.fromkeys(key_cols + ["source_record_hash"])))
        past_keys = previous.select(*key_cols).distinct().withColumn("__was_key", F.lit(1))
        past_payload = previous.select(*list(dict.fromkeys(key_cols + ["source_record_hash"]))).distinct().withColumn("__was_exact", F.lit(1))
        w_key = Window.partitionBy(*key_cols).orderBy(F.col("source_record_hash"))
        w_hash = Window.partitionBy(*(list(dict.fromkeys(key_cols + ["source_record_hash"]))))                       .orderBy(F.col("source_record_hash"))
        events = (incoming
            .withColumn("__rank_key", F.row_number().over(w_key))
            .withColumn("__rank_hash", F.row_number().over(w_hash))
            .join(past_keys, key_cols, "left")
            .join(past_payload, list(dict.fromkeys(key_cols + ["source_record_hash"])), "left")
            .withColumn("__repeated", (F.col("__was_key") == 1) | (F.col("__rank_key") > 1))
            .withColumn("__exact", F.col("__repeated") &
                        ((F.col("__was_exact") == 1) | (F.col("__rank_hash") > 1))))
        row = events.agg(
            F.sum(F.col("__repeated").cast("long")).alias("repeated"),
            F.sum(F.col("__exact").cast("long")).alias("exact")
        ).first()
        rep = int(row["repeated"] or 0)
        exact = int(row["exact"] or 0)
        metrics = (n, rep, exact, rep - exact, n - rep)
    n, rep, exact, conflict, new = metrics
    level = ("INFO_GEO_REPEATED" if by_payload_only and rep else
             "WARN_CONFLICT" if conflict else "WARN_DUPLICATES" if rep else "OK")
    schema = StructType([
        StructField("silver_run_id", StringType()), StructField("bronze_batch_id", StringType()),
        StructField("source_table", StringType()), StructField("rows_received", LongType()),
        StructField("repeated_key_rows", LongType()), StructField("exact_duplicate_rows", LongType()),
        StructField("conflicting_version_rows", LongType()), StructField("new_key_rows", LongType()),
        StructField("alert_level", StringType()), StructField("recorded_at", TimestampType())
    ])
    row_out = [(RUN_ID, BRONZE_BATCH_ID, table, n, rep, exact, conflict, new, level,
                datetime.now(timezone.utc))]
    spark.createDataFrame(row_out, schema).write.format("delta").mode("append").saveAsTable(DUPLICATE_AUDIT)
    print(f"DQ INGESTA {table:<22} lote={n:,} nuevas_claves={new:,} "
          f"duplicados={rep:,} (idénticos={exact:,}, conflictos={conflict:,}) [{level}]")


def process_entity(table, transform_fn):
    raw = read_bronze(table)
    source_rows = raw.count()
    transformed = transform_fn(raw)
    invalid = transformed.filter(key_invalid(transformed, KEYS[table]))
    rejected_rows = invalid.count()
    save_rejects(invalid, table)

    valid = transformed.filter(~key_invalid(transformed, KEYS[table]))
    valid_rows = valid.count()
    audit_duplicate_batch(valid, table, KEYS[table])
    winners = choose_one_per_key(valid, KEYS[table])
    winners = winners.drop(*[
        c for c in winners.columns if c.startswith("__")
    ])
    winners = add_silver_hash(winners)
    resolved_rows = winners.count()
    warnings = winners.filter(F.col("dq_issue_count") > 0).count()

    # Keep old Silver tables and evolve them safely via ALTER TABLE + MERGE.
    upsert_silver(winners, table, KEYS[table])
    persisted_rows = spark.table(silver_name(table)).count()
    audit_table(table, source_rows, rejected_rows, valid_rows,
                valid_rows - resolved_rows, persisted_rows, warnings)
    print(f"OK {table:<21} bronze={source_rows:,} validos={valid_rows:,} "
          f"rechazados={rejected_rows:,} silver={persisted_rows:,} "
          f"alertas={warnings:,}")


In [0]:
# CELDA 3 — Reglas para ocho entidades (Geolocation va en la celda siguiente)

def transform_customers(df):
    df = normalize_text(df, ["customer_id", "customer_unique_id", "customer_zip_code_prefix"])
    df = normalize_text(df, ["customer_city"], "city")
    df = normalize_text(df, ["customer_state"], "upper")
    return add_quality(df, [
        ("MISSING_UNIQUE_CUSTOMER", F.col("customer_unique_id").isNull()),
        ("INVALID_ZIP", ~F.coalesce(F.col("customer_zip_code_prefix").rlike(r"^[0-9]{5}$"), F.lit(False))),
        ("MISSING_CITY", F.col("customer_city").isNull()),
        ("INVALID_STATE", ~F.coalesce(F.col("customer_state").rlike(r"^[A-Z]{2}$"), F.lit(False)))
    ])


def transform_sellers(df):
    df = normalize_text(df, ["seller_id", "seller_zip_code_prefix"])
    df = normalize_text(df, ["seller_city"], "city")
    df = normalize_text(df, ["seller_state"], "upper")
    return add_quality(df, [
        ("INVALID_ZIP", ~F.coalesce(F.col("seller_zip_code_prefix").rlike(r"^[0-9]{5}$"), F.lit(False))),
        ("MISSING_CITY", F.col("seller_city").isNull()),
        ("INVALID_STATE", ~F.coalesce(F.col("seller_state").rlike(r"^[A-Z]{2}$"), F.lit(False)))
    ])


def transform_categories(df):
    df = normalize_text(df, ["product_category_name", "product_category_name_english"], "lower")
    return add_quality(df, [
        ("MISSING_ENGLISH_CATEGORY", F.col("product_category_name_english").isNull())
    ])


def transform_orders(df):
    df = normalize_text(df, ["order_id", "customer_id"])
    df = normalize_text(df, ["order_status"], "lower")
    dates = [
        "order_purchase_timestamp", "order_approved_at",
        "order_delivered_carrier_date", "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
    for c in dates:
        df = df.withColumn(f"__raw_{c}", F.col(c))
    df = safe_cast(df, {c: "TIMESTAMP" for c in dates})

    purchase = F.col("order_purchase_timestamp")
    approved = F.col("order_approved_at")
    carrier = F.col("order_delivered_carrier_date")
    delivered = F.col("order_delivered_customer_date")
    estimated = F.col("order_estimated_delivery_date")

    df = df.withColumn("fecha_inconsistente", (
        (approved < purchase) |
        (carrier < purchase) |
        (carrier < approved) |
        (delivered < approved) |
        (delivered < purchase) |
        (delivered < carrier) |
        (estimated < purchase)
    ))
    df = df.withColumn("fecha_inconsistente", F.coalesce(F.col("fecha_inconsistente"), F.lit(False)))
    df = df.withColumn("is_delivered_late", F.when(
        delivered.isNotNull() & estimated.isNotNull(), delivered > estimated
    ).otherwise(F.lit(None).cast("boolean")))

    statuses = ["created", "approved", "invoiced", "processing", "shipped",
                "delivered", "unavailable", "canceled"]
    rules = [(f"INVALID_{c.upper()}", invalid_conversion(F.col(f"__raw_{c}"), F.col(c)))
             for c in dates]
    rules += [
        ("MISSING_CUSTOMER_ID", F.col("customer_id").isNull()),
        ("INVALID_STATUS", ~F.coalesce(F.col("order_status").isin(statuses), F.lit(False))),
        ("MISSING_PURCHASE_DATE", purchase.isNull()),
        ("APPROVAL_BEFORE_PURCHASE", approved < purchase),
        ("CARRIER_BEFORE_PURCHASE", carrier < purchase),
        ("CARRIER_BEFORE_APPROVAL", carrier < approved),
        ("DELIVERY_BEFORE_APPROVAL", delivered < approved),
        ("DELIVERY_BEFORE_PURCHASE", delivered < purchase),
        ("DELIVERY_BEFORE_CARRIER", delivered < carrier),
        ("ESTIMATE_BEFORE_PURCHASE", estimated < purchase),
        ("DELIVERED_STATUS_NO_DELIVERY_DATE", (F.col("order_status") == "delivered") & delivered.isNull())
    ]
    return add_quality(df, rules)


def transform_items(df):
    df = normalize_text(df, ["order_id", "product_id", "seller_id"])
    typed = {
        "order_item_id": "INT", "price": "DECIMAL(18,2)",
        "freight_value": "DECIMAL(18,2)", "shipping_limit_date": "TIMESTAMP"
    }
    for c in typed:
        df = df.withColumn(f"__raw_{c}", F.col(c))
    df = safe_cast(df, typed)
    rules = [(f"INVALID_{c.upper()}", invalid_conversion(F.col(f"__raw_{c}"), F.col(c)))
             for c in typed]
    rules += [
        ("MISSING_PRODUCT", F.col("product_id").isNull()),
        ("MISSING_SELLER", F.col("seller_id").isNull()),
        ("NON_POSITIVE_ITEM_ID", F.col("order_item_id") <= 0),
        ("MISSING_PRICE", F.col("price").isNull()),
        ("MISSING_FREIGHT", F.col("freight_value").isNull()),
        ("NEGATIVE_PRICE", F.col("price") < 0),
        ("ZERO_PRICE", F.col("price") == 0),
        ("NEGATIVE_FREIGHT", F.col("freight_value") < 0),
        ("MISSING_SHIPPING_LIMIT", F.col("shipping_limit_date").isNull())
    ]
    return add_quality(df, rules)


def transform_payments(df):
    df = normalize_text(df, ["order_id"])
    df = normalize_text(df, ["payment_type"], "lower")
    typed = {
        "payment_sequential": "INT", "payment_installments": "INT",
        "payment_value": "DECIMAL(18,2)"
    }
    for c in typed:
        df = df.withColumn(f"__raw_{c}", F.col(c))
    df = safe_cast(df, typed)
    rules = [(f"INVALID_{c.upper()}", invalid_conversion(F.col(f"__raw_{c}"), F.col(c)))
             for c in typed]
    rules += [
        ("NON_POSITIVE_PAYMENT_SEQUENCE", F.col("payment_sequential") <= 0),
        ("MISSING_PAYMENT_VALUE", F.col("payment_value").isNull()),
        ("MISSING_INSTALLMENTS", F.col("payment_installments").isNull()),
        ("INVALID_INSTALLMENTS", F.col("payment_installments") <= 0),
        ("NEGATIVE_PAYMENT", F.col("payment_value") < 0),
        ("ZERO_PAYMENT", F.col("payment_value") == 0),
        ("MISSING_PAYMENT_TYPE", F.col("payment_type").isNull())
    ]
    return add_quality(df, rules)


def transform_reviews(df):
    df = normalize_text(df, ["review_id", "order_id"])
    typed = {
        "review_score": "INT", "review_creation_date": "TIMESTAMP",
        "review_answer_timestamp": "TIMESTAMP"
    }
    for c in typed:
        df = df.withColumn(f"__raw_{c}", F.col(c))
    df = safe_cast(df, typed)
    df = df.withColumn("has_comment", (
        F.col("review_comment_message").isNotNull() &
        (F.length(F.trim(F.col("review_comment_message"))) > 0)
    ))
    rules = [(f"INVALID_{c.upper()}", invalid_conversion(F.col(f"__raw_{c}"), F.col(c)))
             for c in typed]
    rules += [
        ("INVALID_REVIEW_SCORE", ~F.coalesce(F.col("review_score").between(1, 5), F.lit(False))),
        ("MISSING_REVIEW_CREATION", F.col("review_creation_date").isNull()),
        ("ANSWER_BEFORE_CREATION", F.col("review_answer_timestamp") < F.col("review_creation_date"))
    ]
    # Missing comments/titles are optional, NOT a quality failure.
    return add_quality(df, rules)


def transform_products(df):
    df = normalize_text(df, ["product_id"])
    df = normalize_text(df, ["product_category_name"], "lower")
    typed = {
        "product_name_lenght": "INT", "product_description_lenght": "INT",
        "product_photos_qty": "INT", "product_weight_g": "INT",
        "product_length_cm": "INT", "product_height_cm": "INT",
        "product_width_cm": "INT"
    }
    for c in typed:
        df = df.withColumn(f"__raw_{c}", F.col(c))
    df = safe_cast(df, typed)

    # Enrich only with the unique Silver translation dimension.
    translations = spark.table(silver_name("category_translation")).select(
        "product_category_name", "product_category_name_english"
    )
    df = df.join(translations, on="product_category_name", how="left")

    rules = [(f"INVALID_{c.upper()}", invalid_conversion(F.col(f"__raw_{c}"), F.col(c)))
             for c in typed]
    rules += [
        ("MISSING_CATEGORY", F.col("product_category_name").isNull()),
        ("UNTRANSLATED_CATEGORY", F.col("product_category_name").isNotNull() &
         F.col("product_category_name_english").isNull()),
        ("MISSING_WEIGHT", F.col("product_weight_g").isNull()),
        ("MISSING_LENGTH", F.col("product_length_cm").isNull()),
        ("MISSING_HEIGHT", F.col("product_height_cm").isNull()),
        ("MISSING_WIDTH", F.col("product_width_cm").isNull()),
        ("ZERO_OR_NEGATIVE_WEIGHT", F.col("product_weight_g") <= 0),
        ("INVALID_LENGTH", F.col("product_length_cm") <= 0),
        ("INVALID_HEIGHT", F.col("product_height_cm") <= 0),
        ("INVALID_WIDTH", F.col("product_width_cm") <= 0),
        ("NEGATIVE_PHOTO_COUNT", F.col("product_photos_qty") < 0),
        ("NEGATIVE_NAME_LENGTH", F.col("product_name_lenght") < 0),
        ("NEGATIVE_DESCRIPTION_LENGTH", F.col("product_description_lenght") < 0)
    ]
    return add_quality(df, rules)

TRANSFORMS = {
    "customers": transform_customers,
    "sellers": transform_sellers,
    "category_translation": transform_categories,
    "orders": transform_orders,
    "order_items": transform_items,
    "order_payments": transform_payments,
    "order_reviews": transform_reviews,
    "products": transform_products
}

for name in TABLES:
    if name != "geolocation":
        process_entity(name, TRANSFORMS[name])


In [0]:
# Geolocation — mismo grano ZIP, sin inflación por recargar copias idénticas.
# Mantenemos la multiplicidad del lote original, incluidos duplicados que existían en ese CSV.
# Entre lotes, usamos el mayor número de apariciones por payload (no su suma).
def canonical_geo_observations():
    raw = read_bronze("geolocation")
    raw = normalize_text(raw, ["geolocation_zip_code_prefix"])
    raw = normalize_text(raw, ["geolocation_city"], "city")
    raw = normalize_text(raw, ["geolocation_state"], "upper")
    valid = raw.filter(~key_invalid(raw, KEYS["geolocation"]))
    valid = safe_cast(valid, {"geolocation_lat": "DOUBLE", "geolocation_lng": "DOUBLE"})
    # Un batch legado (filas anteriores a incorporar el linaje) cuenta como lote inicial.
    by_batch = (valid.withColumn("__group_batch", F.coalesce(F.col("__ingest_batch_id"), F.lit("LEGACY")))
        .groupBy("source_record_hash", "__group_batch")
        .agg(F.count("*").cast("long").alias("__copies"),
             *[F.first(c, ignorenulls=False).alias(c) for c in
               ("geolocation_zip_code_prefix", "geolocation_city", "geolocation_state",
                "geolocation_lat", "geolocation_lng")]))
    return (by_batch.groupBy("source_record_hash")
        .agg(F.max("__copies").alias("__weight"),
             *[F.first(c, ignorenulls=False).alias(c) for c in
               ("geolocation_zip_code_prefix", "geolocation_city", "geolocation_state",
                "geolocation_lat", "geolocation_lng")]))

def process_geolocation():
    table = "geolocation"
    raw = read_bronze(table)
    source_rows = raw.count()
    raw = normalize_text(raw, ["geolocation_zip_code_prefix"])
    raw = normalize_text(raw, ["geolocation_city"], "city")
    raw = normalize_text(raw, ["geolocation_state"], "upper")
    invalid = raw.filter(key_invalid(raw, KEYS[table]))
    rejected_rows = invalid.count()
    save_rejects(invalid, table)
    valid = raw.filter(~key_invalid(raw, KEYS[table]))
    valid_rows = valid.count()
    audit_duplicate_batch(valid, table, KEYS[table], by_payload_only=True)

    geo = canonical_geo_observations()
    coord_ok = (F.col("geolocation_lat").between(-90.0, 90.0) &
                F.col("geolocation_lng").between(-180.0, 180.0))
    geo = geo.withColumn("__bad_coordinates", (~F.coalesce(coord_ok, F.lit(False))).cast("long"))
    weighted_lat = F.sum(F.when(coord_ok, F.col("geolocation_lat") * F.col("__weight")))
    weighted_lng = F.sum(F.when(coord_ok, F.col("geolocation_lng") * F.col("__weight")))
    weighted_n = F.sum(F.when(coord_ok, F.col("__weight")))
    groups = (geo.groupBy("geolocation_zip_code_prefix", "geolocation_city", "geolocation_state")
        .agg(F.sum("__weight").alias("frecuencia"),
             F.round(weighted_lat / weighted_n, 6).alias("geolocation_lat"),
             F.round(weighted_lng / weighted_n, 6).alias("geolocation_lng")))
    zip_stats = (geo.groupBy("geolocation_zip_code_prefix")
        .agg(F.sum("__weight").alias("zip_total_observations"),
             F.countDistinct("geolocation_state").alias("zip_distinct_states"),
             F.sum(F.col("__bad_coordinates") * F.col("__weight")).cast("long").alias("zip_invalid_coordinates")))
    window = Window.partitionBy("geolocation_zip_code_prefix").orderBy(
        F.desc("frecuencia"), F.col("geolocation_state").asc_nulls_last(),
        F.col("geolocation_city").asc_nulls_last())
    selected = (groups.withColumn("__rank", F.row_number().over(window))
        .filter(F.col("__rank") == 1).drop("__rank")
        .join(zip_stats, "geolocation_zip_code_prefix", "inner")
        .withColumn("zip_state_ambiguous", F.col("zip_distinct_states") > 1)
        .withColumn("source_record_hash", F.lit(None).cast("string")))
    selected = add_quality(selected, [
        ("INVALID_ZIP", ~F.coalesce(F.col("geolocation_zip_code_prefix").rlike(r"^[0-9]{5}$"), F.lit(False))),
        ("MISSING_CITY", F.col("geolocation_city").isNull()),
        ("INVALID_STATE", ~F.coalesce(F.col("geolocation_state").rlike(r"^[A-Z]{2}$"), F.lit(False))),
        ("AMBIGUOUS_ZIP_STATE", F.col("zip_state_ambiguous")),
        ("INVALID_COORDINATES_IN_ZIP", F.col("zip_invalid_coordinates") > 0)
    ])
    selected = add_silver_hash(selected)
    winners_count = selected.count()
    warnings = selected.filter(F.col("dq_issue_count") > 0).count()
    upsert_silver(selected, table, KEYS[table])
    silver_rows = spark.table(silver_name(table)).count()
    audit_table(table, source_rows, rejected_rows, valid_rows,
                valid_rows - winners_count, silver_rows, warnings)
    print(f"OK geolocation           bronze={source_rows:,} zip_unicos={silver_rows:,} alertas={warnings:,}")

process_geolocation()


In [0]:
# CELDA 5 — Validación integral (clave + reconciliación + banderas + auditoría)
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType
)

checks = []
for table in TABLES:
    df = spark.table(silver_name(table))
    keys = KEYS[table]
    count = df.count()
    unique_count = df.select(*keys).distinct().count()
    bad_key_count = df.filter(key_invalid(df, keys)).count()
    warnings = df.filter(F.col("dq_issue_count") > 0).count()
    valid_state = count == unique_count and bad_key_count == 0
    checks.append((table, count, unique_count, bad_key_count, warnings, "OK" if valid_state else "ERROR"))

validation_schema = StructType([
    StructField("tabla", StringType()),
    StructField("registros_silver", LongType()),
    StructField("claves_unicas", LongType()),
    StructField("claves_invalidas", LongType()),
    StructField("filas_con_alertas", LongType()),
    StructField("estado", StringType())
])
results = spark.createDataFrame(checks, validation_schema)
display(results.orderBy("tabla"))
assert all(r[-1] == "OK" for r in checks), "Fallaron controles de claves Silver"

print("--- Anomalías relevantes ---")
display(
    spark.table(silver_name("orders"))
         .groupBy("dq_issues").count().orderBy(F.desc("count")).limit(12)
)
display(
    spark.table(silver_name("products"))
         .groupBy("dq_issues").count().orderBy(F.desc("count")).limit(12)
)
print("--- Auditoría de esta ejecución ---")
display(spark.table(AUDIT).where(F.col("run_id") == RUN_ID).orderBy("source_table"))
print("SILVER: 9 tablas verificadas. Revisar las banderas WARN antes de Gold.")


In [0]:
# CELDA 6 — Integridad referencial y recuento de problemas conocidos

RELATIONS = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "seller_id", "sellers", "seller_id"),
    ("order_payments", "order_id", "orders", "order_id"),
    ("order_reviews", "order_id", "orders", "order_id")
]

orphan_checks = []
for child, child_col, parent, parent_col in RELATIONS:
    child_keys = (
        spark.table(silver_name(child))
             .select(F.col(child_col).alias("fk"))
             .where(F.col("fk").isNotNull())
             .distinct()
    )
    parent_keys = (
        spark.table(silver_name(parent))
             .select(F.col(parent_col).alias("fk"))
             .distinct()
    )
    orphan_count = child_keys.join(parent_keys, on="fk", how="left_anti").count()
    orphan_checks.append((child, child_col, parent, orphan_count))

orphans_schema = StructType([
    StructField("tabla_hija", StringType()),
    StructField("campo_fk", StringType()),
    StructField("tabla_padre", StringType()),
    StructField("valores_fk_huerfanos", LongType())
])
display(spark.createDataFrame(orphan_checks, orphans_schema))

print("Productos sin categoría:", spark.table(silver_name("products"))
      .where(F.col("product_category_name").isNull()).count())
print("Productos con peso cero:", spark.table(silver_name("products"))
      .where(F.col("product_weight_g") == 0).count())
print("ZIP asociados a más de un estado:", spark.table(silver_name("geolocation"))
      .where("zip_state_ambiguous = true").count())
print("Pedidos con alerta cronológica:", spark.table(silver_name("orders"))
      .where("fecha_inconsistente = true").count())

assert all(r[3] == 0 for r in orphan_checks), (
    "Hay claves foráneas huérfanas. Revisar si son entregas tardías "
    "antes de generar Gold. Las filas se conservaron en Silver."
)
print("CONTROLES REFERENCIALES EXITOSOS")


In [0]:
# CELDA 7 — Consulta rápida de muestras de problemas / cuarentena
display(spark.table(REJECTS).orderBy(F.desc("first_detected_at")).limit(20))
display(spark.table(silver_name("orders")).where("dq_status = 'WARN'").limit(20))
display(spark.table(silver_name("products")).where("dq_status = 'WARN'").limit(20))


In [0]:

from pyspark.sql import functions as F

PRODUCTS = "workspace.olist_lakehouse.silver_products"

# Agregar columna solamente si todavía no existe
if "category_for_analysis" not in spark.table(PRODUCTS).columns:
    spark.sql(f"""
        ALTER TABLE {PRODUCTS}
        ADD COLUMNS (category_for_analysis STRING)
    """)

# Completar la categoría analítica sin modificar las originales
spark.sql(f"""
    UPDATE {PRODUCTS}
    SET category_for_analysis = COALESCE(
        NULLIF(TRIM(product_category_name_english), ''),
        NULLIF(TRIM(product_category_name), ''),
        'Sin categoría'
    )
    WHERE NOT (
        category_for_analysis <=> COALESCE(
            NULLIF(TRIM(product_category_name_english), ''),
            NULLIF(TRIM(product_category_name), ''),
            'Sin categoría'
        )
    )
""")

display(
    spark.table(PRODUCTS)
    .groupBy("category_for_analysis")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
)

print("Categorías analíticas actualizadas.")


In [0]:

from delta.tables import DeltaTable

BRONZE_GEO = "workspace.olist_lakehouse.bronze_geolocation"
SILVER_GEO = "workspace.olist_lakehouse.silver_geolocation"

# 1. Normalizar observaciones originales
geo_raw = (canonical_geo_observations()
    .select(F.col("geolocation_zip_code_prefix").alias("zip"),
            F.col("geolocation_city").alias("city"),
            F.col("geolocation_state").alias("state"),
            F.col("__weight").alias("peso")))

# 2. Frecuencias de cada combinación ZIP / ciudad / estado
geo_counts = (
    geo_raw
    .groupBy("zip", "city", "state")
    .agg(F.sum("peso").alias("n"))
)

# 3. Ubicación seleccionada actualmente en Silver
geo_selected = (
    spark.table(SILVER_GEO)
    .select(
        F.col("geolocation_zip_code_prefix").alias("zip"),
        F.col("geolocation_city").alias("selected_city"),
        F.col("geolocation_state").alias("selected_state")
    )
)

# 4. Calcular respaldo de la ubicación seleccionada
geo_confidence = (
    geo_counts.alias("g")
    .join(geo_selected.alias("s"), on="zip", how="inner")
    .groupBy("zip")
    .agg(
        F.sum("n").alias("zip_reference_observations"),

        F.sum(
            F.when(
                F.col("g.state").eqNullSafe(F.col("s.selected_state")),
                F.col("n")
            ).otherwise(0)
        ).alias("state_support"),

        F.sum(
            F.when(
                F.col("g.state").eqNullSafe(F.col("s.selected_state"))
                & F.col("g.city").eqNullSafe(F.col("s.selected_city")),
                F.col("n")
            ).otherwise(0)
        ).alias("location_support")
    )
    .withColumn(
        "zip_state_support_ratio",
        F.col("state_support") / F.col("zip_reference_observations")
    )
    .withColumn(
        "zip_location_support_ratio",
        F.col("location_support") / F.col("zip_reference_observations")
    )
    .select(
        F.col("zip").alias("geolocation_zip_code_prefix"),
        "zip_reference_observations",
        "zip_state_support_ratio",
        "zip_location_support_ratio"
    )
)

# 5. Incorporar columnas faltantes
nuevas_columnas = {
    "zip_reference_observations": "BIGINT",
    "zip_state_support_ratio": "DOUBLE",
    "zip_location_support_ratio": "DOUBLE"
}

columnas_actuales = spark.table(SILVER_GEO).columns

for nombre, tipo in nuevas_columnas.items():
    if nombre not in columnas_actuales:
        spark.sql(
            f"ALTER TABLE {SILVER_GEO} "
            f"ADD COLUMNS ({nombre} {tipo})"
        )

# 6. Actualizar registros existentes
(
    DeltaTable.forName(spark, SILVER_GEO)
    .alias("t")
    .merge(
        geo_confidence.alias("s"),
        "t.geolocation_zip_code_prefix = s.geolocation_zip_code_prefix"
    )
    .whenMatchedUpdate(condition="NOT (t.zip_reference_observations <=> s.zip_reference_observations) "
                                 "OR NOT (t.zip_state_support_ratio <=> s.zip_state_support_ratio) "
                                 "OR NOT (t.zip_location_support_ratio <=> s.zip_location_support_ratio)", set={
        "zip_reference_observations": "s.zip_reference_observations",
        "zip_state_support_ratio": "s.zip_state_support_ratio",
        "zip_location_support_ratio": "s.zip_location_support_ratio"
    })
    .execute()
)

# 7. Ver los ZIP con conflictos entre estados
display(
    spark.table(SILVER_GEO)
    .filter(F.col("zip_state_ambiguous") == True)
    .select(
        "geolocation_zip_code_prefix",
        "geolocation_city",
        "geolocation_state",
        "zip_distinct_states",
        F.round("zip_state_support_ratio", 4).alias("confianza_estado"),
        F.round("zip_location_support_ratio", 4).alias("confianza_ubicacion")
    )
    .orderBy("confianza_estado")
)

print("Confianza geográfica calculada.")


In [0]:

SILVER_CUSTOMERS = "workspace.olist_lakehouse.silver_customers"
SILVER_SELLERS = "workspace.olist_lakehouse.silver_sellers"

zip_geo = (
    spark.table(SILVER_GEO)
    .select(
        F.col("geolocation_zip_code_prefix").alias("zip")
    )
    .distinct()
)

fuentes = [
    (
        "customers",
        SILVER_CUSTOMERS,
        "customer_zip_code_prefix"
    ),
    (
        "sellers",
        SILVER_SELLERS,
        "seller_zip_code_prefix"
    )
]

resultados_cobertura = []

for nombre, tabla, columna_zip in fuentes:

    df = spark.table(tabla)

    referencias = (
        df.select(
            F.trim(F.col(columna_zip)).alias("zip")
        )
        .filter(F.col("zip").isNotNull())
    )

    sin_match = referencias.join(
        zip_geo,
        on="zip",
        how="left_anti"
    )

    total = df.count()
    faltantes = sin_match.count()
    zip_distintos = sin_match.select("zip").distinct().count()

    resultados_cobertura.append(
        (nombre, total, faltantes, zip_distintos)
    )

    print(
        f"{nombre}: {faltantes:,} registros sin "
        f"correspondencia geográfica"
    )

    display(
        sin_match.groupBy("zip")
        .count()
        .orderBy(F.desc("count"))
        .limit(10)
    )

schema_cobertura = [
    "tabla",
    "registros_totales",
    "registros_sin_match",
    "zip_distintos_sin_match"
]

display(
    spark.createDataFrame(
        resultados_cobertura,
        schema_cobertura
    )
)


In [0]:

products = spark.table(PRODUCTS)
geo = spark.table(SILVER_GEO)

assert products.count() == (spark.table(bronze_name("products"))
    .where(F.col("product_id").isNotNull() & (F.trim("product_id") != ""))
    .select("product_id").distinct().count()), "No coincide grano producto Silver/Bronze"
assert geo.count() == (spark.table(bronze_name("geolocation"))
    .where(F.col("geolocation_zip_code_prefix").isNotNull() &
           (F.trim("geolocation_zip_code_prefix") != ""))
    .select(F.trim("geolocation_zip_code_prefix").alias("zip")).distinct().count()), "No coincide grano ZIP Silver/Bronze"

# La nueva categoría analítica nunca debe ser nula
assert products.filter(
    F.col("category_for_analysis").isNull()
).count() == 0

# Todas las ubicaciones deben contar con estadísticas
assert geo.filter(
    F.col("zip_reference_observations").isNull()
).count() == 0

# La confianza debe estar entre cero y uno
assert geo.filter(
    F.col("zip_state_support_ratio").isNull()
    | F.col("zip_location_support_ratio").isNull()
    | ~F.col("zip_state_support_ratio").between(0, 1)
    | ~F.col("zip_location_support_ratio").between(0, 1)
).count() == 0

print("MEJORAS DE SILVER VALIDADAS")
print("Productos:", products.count())
print("ZIP únicos:", geo.count())
print(
    "Productos sin categoría original:",
    products.filter(
        F.col("product_category_name").isNull()
    ).count()
)
print(
    "ZIP ambiguos:",
    geo.filter(F.col("zip_state_ambiguous")).count()
)


In [0]:

from delta.tables import DeltaTable
from pyspark.sql import functions as F

GEO = "workspace.olist_lakehouse.silver_geolocation"

geo_zip = (
    spark.table(GEO)
    .select(
        F.col("geolocation_zip_code_prefix").alias("zip")
    )
    .distinct()
)

tablas = [
    ("workspace.olist_lakehouse.silver_customers",
     "customer_id", "customer_zip_code_prefix"),
    ("workspace.olist_lakehouse.silver_sellers",
     "seller_id", "seller_zip_code_prefix")
]

for tabla, clave, columna_zip in tablas:

    if "zip_geo_match" not in spark.table(tabla).columns:
        spark.sql(
            f"ALTER TABLE {tabla} "
            "ADD COLUMNS (zip_geo_match BOOLEAN)"
        )

    cobertura = (
        spark.table(tabla)
        .select(
            F.col(clave),
            F.trim(F.col(columna_zip)).alias("zip")
        )
        .join(
            geo_zip.withColumn("zip_geo_match", F.lit(True)),
            on="zip",
            how="left"
        )
        .select(
            clave,
            F.coalesce(
                F.col("zip_geo_match"),
                F.lit(False)
            ).alias("zip_geo_match")
        )
    )

    (
        DeltaTable.forName(spark, tabla)
        .alias("t")
        .merge(
            cobertura.alias("s"),
            f"t.{clave} = s.{clave}"
        )
        .whenMatchedUpdate(condition="NOT (t.zip_geo_match <=> s.zip_geo_match)", set={
            "zip_geo_match": "s.zip_geo_match"
        })
        .execute()
    )

    print(f"\n{tabla}")

    display(
        spark.table(tabla)
        .groupBy("zip_geo_match")
        .count()
    )
